In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed/malware-dataset/Windows_Malware_Final_318_Features.csv


In [2]:
# ============================================================
# CELL 1 — FINAL 60:20:20 STRATIFIED DATA SPLIT
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATA_PATH = "/kaggle/input/datasets/mohdtahasayed/malware-dataset/Windows_Malware_Final_318_Features.csv"

# IMPORTANT:
# Use ONE fixed seed for the entire project.
# If your earlier notebook already defined the project seed,
# replace this with that exact value.
RANDOM_STATE = 42

# ------------------------------------------------------------
# Load final dataset
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert "SHA256" in df.columns, "SHA256 column missing"
assert "Type" in df.columns, "Type column missing"

assert df["SHA256"].is_unique, "Duplicate SHA256 values detected"

print("\nTarget distribution:")
print(df["Type"].value_counts().sort_index())

# ------------------------------------------------------------
# Separate identifier, target and features
# ------------------------------------------------------------

sha = df["SHA256"].copy()
y = df["Type"].copy()

feature_columns = [
    c for c in df.columns
    if c not in ["SHA256", "Type"]
]

X = df[feature_columns].copy()

print("\nFeature information:")
print(f"Number of ML features: {len(feature_columns)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# ------------------------------------------------------------
# STEP 1:
# 80% temporary train+validation
# 20% final test
# ------------------------------------------------------------

X_train_val, X_test, y_train_val, y_test, sha_train_val, sha_test = train_test_split(
    X,
    y,
    sha,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# STEP 2:
# Split remaining 80% into:
# 75% train + 25% validation
#
# 0.80 × 0.75 = 0.60
# 0.80 × 0.25 = 0.20
# ------------------------------------------------------------

X_train, X_val, y_train, y_val, sha_train, sha_val = train_test_split(
    X_train_val,
    y_train_val,
    sha_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# Check sizes
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL SPLIT")
print("=" * 60)

print(f"Train      : {len(X_train):,} samples ({len(X_train)/len(df)*100:.2f}%)")
print(f"Validation : {len(X_val):,} samples ({len(X_val)/len(df)*100:.2f}%)")
print(f"Test       : {len(X_test):,} samples ({len(X_test)/len(df)*100:.2f}%)")
print(f"Total      : {len(X_train)+len(X_val)+len(X_test):,}")

# ------------------------------------------------------------
# SHA256 leakage checks
# ------------------------------------------------------------

train_sha = set(sha_train)
val_sha = set(sha_val)
test_sha = set(sha_test)

print("\n" + "=" * 60)
print("SHA256 LEAKAGE CHECK")
print("=" * 60)

print("Train ∩ Validation :", len(train_sha & val_sha))
print("Train ∩ Test       :", len(train_sha & test_sha))
print("Validation ∩ Test  :", len(val_sha & test_sha))

assert len(train_sha & val_sha) == 0
assert len(train_sha & test_sha) == 0
assert len(val_sha & test_sha) == 0

assert len(train_sha | val_sha | test_sha) == len(df)

print("\n✓ No SHA256 leakage detected")

# ------------------------------------------------------------
# Class distributions
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION")
print("=" * 60)

distribution = pd.DataFrame({
    "Train": y_train.value_counts().sort_index(),
    "Validation": y_val.value_counts().sort_index(),
    "Test": y_test.value_counts().sort_index()
})

distribution["Train_%"] = distribution["Train"] / len(y_train) * 100
distribution["Validation_%"] = distribution["Validation"] / len(y_val) * 100
distribution["Test_%"] = distribution["Test"] / len(y_test) * 100

print(distribution.round(3))

# ------------------------------------------------------------
# Save split identifiers
# ------------------------------------------------------------

split_df = pd.DataFrame({
    "SHA256": pd.concat(
        [sha_train, sha_val, sha_test],
        ignore_index=True
    ),
    "Split": (
        ["train"] * len(sha_train) +
        ["validation"] * len(sha_val) +
        ["test"] * len(sha_test)
    )
})

SPLIT_PATH = "/kaggle/working/Malware_60_20_20_Split.csv"

split_df.to_csv(SPLIT_PATH, index=False)

print(f"\n✓ Split assignments saved to:")
print(SPLIT_PATH)

# ------------------------------------------------------------
# Save individual datasets
# ------------------------------------------------------------

train_df = pd.concat(
    [sha_train.reset_index(drop=True),
     y_train.reset_index(drop=True),
     X_train.reset_index(drop=True)],
    axis=1
)

val_df = pd.concat(
    [sha_val.reset_index(drop=True),
     y_val.reset_index(drop=True),
     X_val.reset_index(drop=True)],
    axis=1
)

test_df = pd.concat(
    [sha_test.reset_index(drop=True),
     y_test.reset_index(drop=True),
     X_test.reset_index(drop=True)],
    axis=1
)

train_df.to_csv(
    "/kaggle/working/Malware_Train_60.csv",
    index=False
)

val_df.to_csv(
    "/kaggle/working/Malware_Validation_20.csv",
    index=False
)

test_df.to_csv(
    "/kaggle/working/Malware_Test_20.csv",
    index=False
)

print("\nSaved:")
print("  Malware_Train_60.csv")
print("  Malware_Validation_20.csv")
print("  Malware_Test_20.csv")

Dataset loaded successfully
Shape: (29489, 320)

Target distribution:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64

Feature information:
Number of ML features: 318
X shape: (29489, 318)
y shape: (29489,)

FINAL SPLIT
Train      : 17,693 samples (60.00%)
Validation : 5,898 samples (20.00%)
Test       : 5,898 samples (20.00%)
Total      : 29,489

SHA256 LEAKAGE CHECK
Train ∩ Validation : 0
Train ∩ Test       : 0
Validation ∩ Test  : 0

✓ No SHA256 leakage detected

CLASS DISTRIBUTION
      Train  Validation  Test  Train_%  Validation_%  Test_%
Type                                                        
0      1127         375   375    6.370         6.358   6.358
1      3013        1004  1005   17.029        17.023  17.040
2      2785         929   929   15.741        15.751  15.751
3      2973         991   991   16.803        16.802  16.802
4      3046        1015  1015   17.216        17.209  17.209
5      2530         844   843 

In [3]:
# ============================================================
# CELL 2 — PREPARE DATA FOR LAZYPREDICT
# ============================================================

import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TRAIN_PATH = "/kaggle/working/Malware_Train_60.csv"
VAL_PATH   = "/kaggle/working/Malware_Validation_20.csv"
TEST_PATH  = "/kaggle/working/Malware_Test_20.csv"

# ------------------------------------------------------------
# Load splits
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Loaded datasets:")
print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

# ------------------------------------------------------------
# Separate identifiers, targets and features
# ------------------------------------------------------------

DROP_COLUMNS = ["SHA256", "Type"]

feature_columns = [
    c for c in train_df.columns
    if c not in DROP_COLUMNS
]

X_train = train_df[feature_columns].copy()
X_val   = val_df[feature_columns].copy()
X_test  = test_df[feature_columns].copy()

y_train = train_df["Type"].copy()
y_val   = val_df["Type"].copy()
y_test  = test_df["Type"].copy()

print("\nFeature count:", len(feature_columns))

# ------------------------------------------------------------
# Identify binary and non-binary features
# ------------------------------------------------------------

binary_features = []
nonbinary_features = []

for col in feature_columns:
    unique_values = train_df[col].dropna().unique()

    if len(unique_values) <= 2 and set(unique_values).issubset({0, 1}):
        binary_features.append(col)
    else:
        nonbinary_features.append(col)

print("\nFeature types:")
print("Binary features    :", len(binary_features))
print("Non-binary features:", len(nonbinary_features))

# ------------------------------------------------------------
# Preprocessing
#
# Same methodology used previously:
#
# Binary:
#     passthrough
#
# Non-binary:
#     log1p
#     StandardScaler
#
# IMPORTANT:
# All transformations are fitted ONLY on training data.
# ------------------------------------------------------------

X_train_processed = X_train.copy()
X_val_processed   = X_val.copy()
X_test_processed  = X_test.copy()

# ------------------------------------------------------------
# Non-binary features
# ------------------------------------------------------------

for col in nonbinary_features:

    # Make sure values are non-negative before log1p.
    # This dataset's selected PE features are expected to be
    # non-negative, but we verify explicitly.

    train_min = X_train_processed[col].min()

    if train_min < 0:
        raise ValueError(
            f"Negative value found in {col}: min={train_min}"
        )

# Apply log1p
X_train_nonbinary = np.log1p(
    X_train_processed[nonbinary_features].astype(np.float32)
)

X_val_nonbinary = np.log1p(
    X_val_processed[nonbinary_features].astype(np.float32)
)

X_test_nonbinary = np.log1p(
    X_test_processed[nonbinary_features].astype(np.float32)
)

# ------------------------------------------------------------
# Fit scaler ONLY on training data
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_nonbinary)
X_val_scaled   = scaler.transform(X_val_nonbinary)
X_test_scaled  = scaler.transform(X_test_nonbinary)

# ------------------------------------------------------------
# Put scaled values back
# ------------------------------------------------------------

X_train_processed[nonbinary_features] = X_train_scaled
X_val_processed[nonbinary_features]   = X_val_scaled
X_test_processed[nonbinary_features]  = X_test_scaled

# ------------------------------------------------------------
# Convert to float32 to reduce memory usage
# ------------------------------------------------------------

X_train_processed = X_train_processed.astype(np.float32)
X_val_processed   = X_val_processed.astype(np.float32)
X_test_processed  = X_test_processed.astype(np.float32)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

print("\nProcessed shapes:")
print("Train      :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test       :", X_test_processed.shape)

print("\nNaN check:")
print("Train      :", X_train_processed.isna().sum().sum())
print("Validation :", X_val_processed.isna().sum().sum())
print("Test       :", X_test_processed.isna().sum().sum())

print("\nInfinite check:")
print("Train      :", np.isinf(X_train_processed.to_numpy()).sum())
print("Validation :", np.isinf(X_val_processed.to_numpy()).sum())
print("Test       :", np.isinf(X_test_processed.to_numpy()).sum())

assert X_train_processed.shape == (len(train_df), 318)
assert X_val_processed.shape == (len(val_df), 318)
assert X_test_processed.shape == (len(test_df), 318)

assert X_train_processed.isna().sum().sum() == 0
assert X_val_processed.isna().sum().sum() == 0
assert X_test_processed.isna().sum().sum() == 0

assert np.isinf(X_train_processed.to_numpy()).sum() == 0
assert np.isinf(X_val_processed.to_numpy()).sum() == 0
assert np.isinf(X_test_processed.to_numpy()).sum() == 0

print("\n✓ LazyPredict input data prepared successfully.")

Loaded datasets:
Train      : (17693, 320)
Validation : (5898, 320)
Test       : (5898, 320)

Feature count: 318

Feature types:
Binary features    : 227
Non-binary features: 91

Processed shapes:
Train      : (17693, 318)
Validation : (5898, 318)
Test       : (5898, 318)

NaN check:
Train      : 0
Validation : 0
Test       : 0

Infinite check:
Train      : 0
Validation : 0
Test       : 0

✓ LazyPredict input data prepared successfully.


In [4]:
# ============================================================
# CELL 3 — CHECK LAZYPREDICT
# ============================================================

try:
    import lazypredict
    from lazypredict.Supervised import LazyClassifier

    print("LazyPredict is installed.")
    print("Version:", getattr(lazypredict, "__version__", "unknown"))

except ImportError:
    print("LazyPredict is NOT installed.")
    print("Installing...")
    
    !pip install -q lazypredict

    print("\nInstallation complete.")

LazyPredict is NOT installed.
Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.9 MB/s eta 0:00:00

Installation complete.


In [5]:
# ============================================================
# CELL 4 — LazyPredict Baseline Model Comparison
# ============================================================

import time
import warnings
warnings.filterwarnings("ignore")

from lazypredict.Supervised import LazyClassifier

print("Starting LazyPredict baseline...")
print(f"Training samples   : {X_train_processed.shape[0]}")
print(f"Validation samples : {X_val_processed.shape[0]}")
print(f"Features           : {X_train_processed.shape[1]}")
print()

start_time = time.time()

clf = LazyClassifier(
    verbose=0,
    ignore_warnings=True,
    custom_metric=None
)

models, predictions = clf.fit(
    X_train_processed,
    X_val_processed,
    y_train,
    y_val
)

elapsed = time.time() - start_time

print(f"\nLazyPredict completed in {elapsed/60:.2f} minutes.")
print(f"Models evaluated: {len(models)}")

display(models)

Starting LazyPredict baseline...
Training samples   : 17693
Validation samples : 5898
Features           : 318


LazyPredict completed in 50.21 minutes.
Models evaluated: 28


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
XGBClassifier,0.889115,0.889834,0.987127,0.890408,0.893626,0.889115,5.775741
LGBMClassifier,0.883520,0.885164,0.987288,0.884985,0.888704,0.883520,5.337936
CatBoostClassifier,0.879620,0.880824,0.985572,0.881047,0.885287,0.879620,64.828760
RandomForestClassifier,0.878942,0.877355,0.983325,0.880513,0.883702,0.878942,4.470331
BaggingClassifier,0.875890,0.874854,0.974795,0.876348,0.877149,0.875890,4.032204
DecisionTreeClassifier,0.863852,0.860723,0.920336,0.863895,0.864003,0.863852,0.745955
ExtraTreesClassifier,0.861648,0.856812,0.977966,0.863247,0.866021,0.861648,5.001655
KNeighborsClassifier,0.820617,0.811682,0.951783,0.822524,0.825586,0.820617,2.195844
ExtraTreeClassifier,0.813496,0.798356,0.891062,0.813863,0.814541,0.813496,0.213709
